# WSZYSTKO RAZEM

In [5]:
def pobierz_dane_gus():
    import pandas as pd
    from datetime import datetime


    ## 1. CPI
    
    url = r"C:\Users\48503\Desktop\Gus_2\surowe\cpi.xlsx"

    df = pd.read_excel(url)
    df = df.loc[df["Sposób prezentacji"] == "Poprzedni miesiąc = 100"]
    df= df[["Rok", "Miesiąc", "Wartość"]]
    df["Wartość"] = round(df["Wartość"],2)
    df.rename(columns={"Wartość": "CPI_r_r"}, inplace=True)

    df["DATA_DATE"] = (
        pd.to_datetime(
            df["Rok"].astype(str) + "-" + df["Miesiąc"].astype(str) + "-01"
        )
        + pd.offsets.MonthEnd(0)
    )
    df = df[["DATA_DATE", "CPI_r_r"]]
    df = df.loc[df["DATA_DATE"] > "2021-12-31"]
    df = df.sort_values("DATA_DATE", ascending=True)
    df.fillna(0, inplace=True)
    
    ## 2. Pobranie danych o wynagrodzeniach
    
    url_2 = (
        r"C:\Users\48503\Desktop\Gus_2\surowe\bezrobocie_wynagrodzenia.xlsx"
    )

    df_2 = pd.read_excel(
        url_2,
        sheet_name="WYNAGR. I ŚWIADCZ. SPOŁ",
        header=None
    )

    mask = (
        df_2.iloc[:, 1]
        .astype(str)
        .str.contains(
            "przeciętne miesięczne nominalne wynagrodzenie",
            case=False,
            na=False
        )
    )

    if not mask.any():
        raise ValueError("Nie znaleziono sekcji wynagrodzeń w danych GUS.")

    start_idx = mask.idxmax()

    rows = {
        "wwyn_zl": df_2.iloc[start_idx],
        "wyn_r/r": df_2.iloc[start_idx + 1],
        "wyn_m/m": df_2.iloc[start_idx + 2],
    }

    years = df_2.iloc[3, 3:]
    months = df_2.iloc[4, 3:]

    records = []

    for col in years.index:
        year = years[col]
        month = months[col]

        if pd.notna(year) and pd.notna(month):
            date = (
                pd.to_datetime(f"{int(year)}-{int(month)}-01")
                + pd.offsets.MonthEnd(0)
            )

            record = {"DATA_DATE": date}

            for name, row in rows.items():
                val = row[col]
                record[name] = None if val in [".", "-", None] else val

            records.append(record)

    df_out = (
        pd.DataFrame(records)
        .sort_values("DATA_DATE")
        .reset_index(drop=True)
    )

    df_out = df_out.loc[df_out["DATA_DATE"] > "2021-12-31"]
    df_out["wyn_r/r"] = round(df_out["wyn_r/r"] - 100.0,2)
    df_out["wyn_m/m"] = round(df_out["wyn_m/m"] - 100.0,2)
    df_out.fillna(0, inplace=True)
    df_do_zapisania = df_out.merge(df, on="DATA_DATE", how="left")
    
    
    # 3. Dane o bezrobociu
    
    url_3 = (
        r"C:\Users\48503\Desktop\Gus_2\surowe\bezrobocie_wynagrodzenia.xlsx"
    )

    df_b = pd.read_excel(
        url_3,
        sheet_name="RYNEK PRACY",
        header=None
    )

    mask = (
        df_b.iloc[:, 1]
        .astype(str)
        .str.contains(
            "stopa bezrobocia rejestrowanego",
            case=False,
            na=False
        )
    )

    if not mask.any():
        raise ValueError("Nie znaleziono sekcji 'Stopa bezrobocia rejestrowanego'")

    start_idx = mask.idxmax()

    rows = {
        "st_bezr": df_b.iloc[start_idx ],  # w %
        "st_bezr_2": df_b.iloc[start_idx + 1],     # r/r
        "st_bezr_3": df_b.iloc[start_idx + 2],     # m/m
    }

    years = df_b.iloc[3, 3:]
    months = df_b.iloc[4, 3:]

    records = []

    for col in years.index:
        year = years[col]
        month = months[col]

        if pd.notna(year) and pd.notna(month):
            date = (
                pd.to_datetime(f"{int(year)}-{int(month)}-01")
                + pd.offsets.MonthEnd(0)
            )

            record = {"DATA_DATE": date}

            for name, row in rows.items():
                val = row[col]
                record[name] = None if val in [".", "-", None] else val

            records.append(record)

    df_bezr = (
        pd.DataFrame(records)
        .sort_values("DATA_DATE")
        .reset_index(drop=True)
    )

    df_bezr = df_bezr.loc[df_bezr["DATA_DATE"] > "2021-12-31"]
    df_bezr["st_bezr"] = df_bezr["st_bezr"].fillna(df_bezr["st_bezr_2"])
    df_bezr.drop(columns=["st_bezr_2","st_bezr_3"], inplace=True)
    df_bezr.fillna(0, inplace=True)
    
    df_do_zapisania = df_do_zapisania.merge(df_bezr, on="DATA_DATE", how="left")

    today = datetime.today().strftime("%Y-%m-%d")
    filename = f"dane_gus_{today}.csv"

    df_do_zapisania.to_csv(filename, index=False, encoding="utf-8-sig")

    return print(f"Dane GUS zostały zapisane do dane_gus_{today}.csv")


pobierz_dane_gus()

Dane GUS zostały zapisane do dane_gus_2026-01-19.csv
